# LeetCode 1376 — Time Needed to Inform All Employees bằng Python

Link đề: https://leetcode.com/problems/time-needed-to-inform-all-employees/ · Độ khó: **Medium** · Chủ đề: Tree, Depth-First Search, Breadth-First Search

Skeleton LeetCode:

```python
class Solution:
    def numOfMinutes(self, n: int, headID: int, manager: list[int], informTime: list[int]) -> int:
        ...
```

## 1. Mục tiêu học

1. Đọc được một **cây cho dưới dạng mảng cha** (`manager[i]` = cha của `i`) và biết khi nào cần đổi sang **danh sách con** (adjacency list).
2. Hiểu đáp án là **đường đi có trọng số lớn nhất từ gốc đến lá**, trọng số là `informTime` của các nút **quản lý** trên đường.
3. Cài đặt bốn cách: đi ngược lên không nhớ (brute force) → đi ngược lên **có memo** → **DFS** từ gốc → **BFS** từ gốc.
4. Nắm kỹ thuật tránh `RecursionError` với cây dạng "dây" `10^5` tầng.
5. Biết phân biệt "thời điểm nhận tin" (cộng `informTime` của **cha**) với "thời gian một người cần để báo tin".

## 2. Đề bài & ví dụ

Công ty có `n` nhân viên đánh số `0..n−1`; người đứng đầu là `headID`. Mỗi nhân viên `i` có một quản lý trực tiếp `manager[i]` (`manager[headID] = −1`). Quan hệ quản lý tạo thành một **cây**.

Người đứng đầu muốn báo một tin khẩn: họ báo cho các cấp dưới trực tiếp, những người này lại báo cho cấp dưới của mình, v.v. Nhân viên `i` cần `informTime[i]` phút để báo cho **tất cả** cấp dưới trực tiếp (sau `informTime[i]` phút, mọi cấp dưới trực tiếp đều biết tin và bắt đầu lan truyền). Trả về số phút cần để **mọi** nhân viên biết tin.

- **Ví dụ 1:** `n = 1, headID = 0, manager = [-1], informTime = [0]` → `0`.
- **Ví dụ 2:** `n = 6, headID = 2, manager = [2,2,-1,2,2,2], informTime = [0,0,1,0,0,0]` → `1`.

Ví dụ bổ sung (xuất hiện trong phiên bản cũ của đề — đối chiếu lại đề gốc; đã kiểm chứng bằng brute force ở mục 12):
- `n = 7, headID = 6, manager = [1,2,3,4,5,6,-1], informTime = [0,6,5,4,3,2,1]` → `21` (dây 6 → 5 → … → 0).
- `n = 15, headID = 0`, cây nhị phân đầy đủ, mọi quản lý có `informTime = 1` → `3`.

**Ràng buộc:** `1 <= n <= 10^5`; `0 <= headID < n`; `manager.length == informTime.length == n`; `0 <= manager[i] < n` (trừ `manager[headID] == -1`); `0 <= informTime[i] <= 1000`; `informTime[i] == 0` nếu `i` không có cấp dưới; đảm bảo mọi người đều được báo tin.

In [1]:
import random
import sys
import time
from collections import deque, defaultdict

EXAMPLES = [
    (1, 0, [-1], [0], 0),
    (6, 2, [2, 2, -1, 2, 2, 2], [0, 0, 1, 0, 0, 0], 1),
    (7, 6, [1, 2, 3, 4, 5, 6, -1], [0, 6, 5, 4, 3, 2, 1], 21),
    (15, 0, [-1, 0, 0, 1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 6], [1] * 7 + [0] * 8, 3),
]
for n, head, mgr, it, want in EXAMPLES:
    print(f"n={n}, head={head} -> mong đợi {want}")

n=1, head=0 -> mong đợi 0
n=6, head=2 -> mong đợi 1
n=7, head=6 -> mong đợi 21
n=15, head=0 -> mong đợi 3


## 3. Kiến thức nền

### 3.1 Cây dạng mảng cha → danh sách con

`manager[i] = p` nghĩa là có cạnh `p → i`. Muốn đi **từ gốc xuống** ta cần biết con của mỗi nút: duyệt một lượt và `children[p].append(i)`. `defaultdict(list)` hoặc `[[] for _ in range(n)]` đều được (đừng dùng `[[]] * n` — mọi phần tử trỏ cùng một list!).

In [2]:
def build_children(n, manager):
    children = [[] for _ in range(n)]
    for i, p in enumerate(manager):
        if p != -1:
            children[p].append(i)
    return children

n, head, mgr, it, _ = EXAMPLES[3]
ch = build_children(n, mgr)
print("con của 0:", ch[0], "| con của 1:", ch[1], "| con của 7:", ch[7])

bad = [[]] * 3
bad[0].append(99)
print("bẫy [[]] * 3:", bad)

con của 0: [1, 2] | con của 1: [3, 4] | con của 7: []
bẫy [[]] * 3: [[99], [99], [99]]


### 3.2 BFS bằng `deque`

`deque.popleft()` là `O(1)` (khác `list.pop(0)` là `O(n)`). BFS duyệt theo tầng, rất hợp để lan truyền "thời điểm nhận tin" từ cha sang con.

In [3]:
q = deque([0])
order = []
while q:
    u = q.popleft()
    order.append(u)
    q.extend(ch[u])
print("thứ tự BFS:", order)

thứ tự BFS: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]


### 3.3 DFS đệ quy, `sys.setrecursionlimit`, và DFS bằng stack

Cây có thể là một "dây" dài `10^5` nút ⇒ đệ quy sâu `10^5` tầng, vượt giới hạn mặc định (~1000). Có thể tăng giới hạn, nhưng an toàn nhất là dùng **stack** tường minh.

In [4]:
print("giới hạn mặc định:", sys.getrecursionlimit())
def depth_iter(children, root):
    best, stack = 0, [(root, 0)]
    while stack:
        u, d = stack.pop()
        best = max(best, d)
        for v in children[u]:
            stack.append((v, d + 1))
    return best
print("độ sâu cây ví dụ 15 nút:", depth_iter(ch, 0))

giới hạn mặc định: 1000
độ sâu cây ví dụ 15 nút: 3


### 3.4 Memo hóa (ghi nhớ) khi đi ngược lên cha

Nếu đã biết `t[p]` = thời điểm `p` nhận tin, thì con `i` của `p` nhận tin lúc `t[p] + informTime[p]`. Lưu `t` vào mảng để **mỗi nút chỉ tính một lần**. Đi ngược lên bằng vòng lặp, gom các nút chưa biết vào một list rồi điền ngược lại.

In [5]:
memo = {}
def fib(k):
    if k < 2:
        return k
    if k not in memo:
        memo[k] = fib(k - 1) + fib(k - 2)
    return memo[k]
print("fib(80) =", fib(80), "| số giá trị đã nhớ:", len(memo))

fib(80) = 23416728348467685 | số giá trị đã nhớ: 79


## 4. Ý tưởng nền tảng / trực giác

Xét cây (ví dụ tự dựng):

```
           0 (it=4)
         /        \
     1 (it=2)    2 (it=1)
       |           |
     3 (it=0)    4 (it=7)
                   |
                 5 (it=0)
```

- `0` biết tin lúc 0. Sau `informTime[0] = 4` phút, cả `1` và `2` biết tin (lúc 4).
- `1` báo `3` mất 2 phút ⇒ `3` biết lúc `4 + 2 = 6`.
- `2` báo `4` mất 1 phút ⇒ `4` biết lúc 5; `4` báo `5` mất 7 phút ⇒ `5` biết lúc 12.

Đáp án = `max(6, 12) = 12`. Quy luật:

> **thời điểm nhận tin của `i` = tổng `informTime` của mọi quản lý trên đường từ `headID` tới `i` (không tính chính `i`).**

Đáp án là max trên mọi nhân viên — thực chất chỉ cần xét **lá**, vì lá luôn nhận tin muộn hơn hoặc bằng tổ tiên của nó.

In [6]:
n4 = 6
mgr4 = [-1, 0, 0, 1, 2, 4]
it4 = [4, 2, 1, 0, 7, 0]
print("ví dụ mục 4:", n4, mgr4, it4, "-> kỳ vọng 12")

ví dụ mục 4: 6 [-1, 0, 0, 1, 2, 4] [4, 2, 1, 0, 7, 0] -> kỳ vọng 12


## 5. Cách 1 — Brute Force / baseline: đi ngược lên từ mỗi nhân viên, không nhớ

**Ý tưởng:** với mỗi nhân viên `i`, lần theo `manager` lên tới `headID`, cộng `informTime` của từng quản lý gặp trên đường. Lấy max.

In [7]:
def minutes_brute(n, headID, manager, informTime):
    best = 0
    for i in range(n):
        t, cur = 0, i
        while manager[cur] != -1:
            cur = manager[cur]
            t += informTime[cur]          # cộng thời gian của QUẢN LÝ
        best = max(best, t)
    return best

for n, head, mgr, it, want in EXAMPLES:
    print(minutes_brute(n, head, mgr, it), "(mong đợi", want, ")")
print("ví dụ mục 4:", minutes_brute(n4, 0, mgr4, it4))

0 (mong đợi 0 )
1 (mong đợi 1 )
21 (mong đợi 21 )
3 (mong đợi 3 )
ví dụ mục 4: 12


**Độ phức tạp:** mỗi nhân viên đi lên tối đa `H` bước (chiều cao cây) ⇒ `O(n·H)`; cây dạng dây `H = n` ⇒ `O(n²) = 10^{10}` — quá chậm. Bộ nhớ `O(1)` thêm.

**Nút thắt:** các nhân viên chung quản lý **đi lại cùng một đoạn đường** lên gốc nhiều lần. Ta muốn tính thời điểm của mỗi nút **đúng một lần**.

In [8]:
for n in [1000, 2000, 4000]:
    mgr_chain = [i + 1 for i in range(n - 1)] + [-1]      # dây: 0 <- 1 <- ... <- n-1 (gốc)
    it_chain = [0] + [1] * (n - 1)
    t0 = time.perf_counter()
    minutes_brute(n, n - 1, mgr_chain, it_chain)
    print(f"dây n={n}: {time.perf_counter() - t0:.3f}s")

dây n=1000: 0.019s
dây n=2000: 0.078s


dây n=4000: 0.325s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** gọi `t[i]` = thời điểm `i` nhận tin. Ta có quan hệ truy hồi theo cạnh cây:

- `t[headID] = 0`
- `t[i] = t[manager[i]] + informTime[manager[i]]`

Đáp án `= max(t[i])`. Mỗi `t[i]` chỉ phụ thuộc cha ⇒ tính theo **bất kỳ thứ tự nào mà cha đứng trước con** (BFS/DFS từ gốc), hoặc tính lười **từ dưới lên với memo**.

Một dạng tương đương **từ dưới lên**: `f(u)` = thời gian để toàn bộ cây con gốc `u` biết tin kể từ lúc `u` biết = `informTime[u] + max(f(con))` (0 nếu là lá). Đáp án `= f(headID)`.

**Quy tắc "nếu… thì…":**
- Nếu đã biết `t[cha]` thì `t[con]` tính ngay trong `O(1)`.
- Nếu `t[i]` đã có trong memo thì **dừng đi lên** ngay tại đó.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Đi ngược lên quản lý có memo

**Từng đoạn:**
1. `t = [-1] * n`, `t[headID] = 0`.
2. Với mỗi `i`: đi lên, gom các nút **chưa biết** vào `path` cho tới khi gặp nút đã biết `cur`.
3. Điền ngược: từ nút gần `cur` nhất trở xuống, `t[x] = t[manager[x]] + informTime[manager[x]]`.

In [9]:
def minutes_memo(n, headID, manager, informTime):
    t = [-1] * n
    t[headID] = 0
    for i in range(n):
        path = []
        cur = i
        while t[cur] == -1:          # đi lên tới nút đã biết
            path.append(cur)
            cur = manager[cur]
        for x in reversed(path):     # điền từ trên xuống
            p = manager[x]
            t[x] = t[p] + informTime[p]
    return max(t)

for n, head, mgr, it, want in EXAMPLES:
    print(minutes_memo(n, head, mgr, it), "(mong đợi", want, ")")

0 (mong đợi 0 )
1 (mong đợi 1 )
21 (mong đợi 21 )
3 (mong đợi 3 )


Code nộp LeetCode (Cách 2):

In [10]:
class Solution:
    def numOfMinutes(self, n: int, headID: int, manager: list[int], informTime: list[int]) -> int:
        t = [-1] * n
        t[headID] = 0
        for i in range(n):
            path, cur = [], i
            while t[cur] == -1:
                path.append(cur)
                cur = manager[cur]
            for x in reversed(path):
                p = manager[x]
                t[x] = t[p] + informTime[p]
        return max(t)

print([Solution().numOfMinutes(n, h, m, it) for n, h, m, it, _ in EXAMPLES])

[0, 1, 21, 3]


**Độ phức tạp Cách 2:** mỗi nút được thêm vào `path` **đúng một lần** trong toàn bộ thuật toán (sau đó `t` đã biết) ⇒ `O(n)` thời gian; `O(n)` bộ nhớ. Không cần dựng danh sách con.

### 7.2 Cách 3 — DFS từ head trên danh sách con

**Từng đoạn:** dựng `children`; DFS (đệ quy) theo dạng từ dưới lên `f(u) = informTime[u] + max(f(v) for v in children[u])`. Ở đây viết bản đệ quy để dễ đọc, kèm tăng giới hạn đệ quy.

In [11]:
sys.setrecursionlimit(300000)

def minutes_dfs(n, headID, manager, informTime):
    children = build_children(n, manager)
    def f(u):
        if not children[u]:
            return 0
        return informTime[u] + max(f(v) for v in children[u])
    return f(headID)

for n, head, mgr, it, want in EXAMPLES:
    print(minutes_dfs(n, head, mgr, it), "(mong đợi", want, ")")

0 (mong đợi 0 )
1 (mong đợi 1 )
21 (mong đợi 21 )
3 (mong đợi 3 )


Code nộp LeetCode (Cách 3) — dùng DFS **top-down bằng stack** để an toàn với cây sâu `10^5`:

In [12]:
class Solution:
    def numOfMinutes(self, n: int, headID: int, manager: list[int], informTime: list[int]) -> int:
        children = [[] for _ in range(n)]
        for i, p in enumerate(manager):
            if p != -1:
                children[p].append(i)
        best = 0
        stack = [(headID, 0)]                 # (nhân viên, thời điểm nhận tin)
        while stack:
            u, t = stack.pop()
            best = max(best, t)
            nt = t + informTime[u]
            for v in children[u]:
                stack.append((v, nt))
        return best

print([Solution().numOfMinutes(n, h, m, it) for n, h, m, it, _ in EXAMPLES])

[0, 1, 21, 3]


**Độ phức tạp Cách 3:** `O(n)` thời gian (dựng `children` + duyệt), `O(n)` bộ nhớ.

### 7.3 Cách 4 — BFS từ head (lan truyền theo tầng)

BFS mô phỏng đúng "tin lan ra": lấy một người khỏi hàng đợi, mọi cấp dưới trực tiếp nhận tin lúc `t[u] + informTime[u]`.

In [13]:
def minutes_bfs(n, headID, manager, informTime):
    children = build_children(n, manager)
    t = [0] * n
    best = 0
    q = deque([headID])
    while q:
        u = q.popleft()
        best = max(best, t[u])
        for v in children[u]:
            t[v] = t[u] + informTime[u]
            q.append(v)
    return best

for n, head, mgr, it, want in EXAMPLES:
    print(minutes_bfs(n, head, mgr, it), "(mong đợi", want, ")")

0 (mong đợi 0 )
1 (mong đợi 1 )
21 (mong đợi 21 )
3 (mong đợi 3 )


Code nộp LeetCode (Cách 4):

In [14]:
class Solution:
    def numOfMinutes(self, n: int, headID: int, manager: list[int], informTime: list[int]) -> int:
        children = defaultdict(list)
        for i, p in enumerate(manager):
            children[p].append(i)
        t = [0] * n
        q = deque([headID])
        best = 0
        while q:
            u = q.popleft()
            if t[u] > best:
                best = t[u]
            for v in children[u]:
                t[v] = t[u] + informTime[u]
                q.append(v)
        return best

print([Solution().numOfMinutes(n, h, m, it) for n, h, m, it, _ in EXAMPLES])

[0, 1, 21, 3]


**Độ phức tạp Cách 4:** `O(n)` thời gian, `O(n)` bộ nhớ.

Lưu ý: BFS **không** cần ưu tiên theo thời gian (không phải Dijkstra) vì mỗi nút chỉ có **một** đường từ gốc — thời điểm nhận tin xác định duy nhất ngay khi xét cha.

## 8. TRACE từng bước

Trace Cách 4 trên ví dụ mục 4 (`manager = [-1,0,0,1,2,4]`, `informTime = [4,2,1,0,7,0]`):

| Lấy ra u | t[u] | informTime[u] | con nhận tin | hàng đợi sau | best |
|---|---|---|---|---|---|
| 0 | 0 | 4 | 1 → 4, 2 → 4 | [1, 2] | 0 |
| 1 | 4 | 2 | 3 → 6 | [2, 3] | 4 |
| 2 | 4 | 1 | 4 → 5 | [3, 4] | 4 |
| 3 | 6 | 0 | – | [4] | 6 |
| 4 | 5 | 7 | 5 → 12 | [5] | 6 |
| 5 | 12 | 0 | – | [] | **12** |

In [15]:
def trace_bfs(n, headID, manager, informTime):
    children = build_children(n, manager)
    t = [0] * n
    best = 0
    q = deque([headID])
    while q:
        u = q.popleft()
        best = max(best, t[u])
        got = []
        for v in children[u]:
            t[v] = t[u] + informTime[u]
            q.append(v)
            got.append(f"{v}->{t[v]}")
        print(f"u={u} | t[u]={t[u]:2d} | informTime={informTime[u]} | con: {', '.join(got) or '-':14s} | queue={list(q)} | best={best}")
    return best

print("kết quả:", trace_bfs(n4, 0, mgr4, it4))

u=0 | t[u]= 0 | informTime=4 | con: 1->4, 2->4     | queue=[1, 2] | best=0
u=1 | t[u]= 4 | informTime=2 | con: 3->6           | queue=[2, 3] | best=4
u=2 | t[u]= 4 | informTime=1 | con: 4->5           | queue=[3, 4] | best=4
u=3 | t[u]= 6 | informTime=0 | con: -              | queue=[4] | best=6
u=4 | t[u]= 5 | informTime=7 | con: 5->12          | queue=[5] | best=6
u=5 | t[u]=12 | informTime=0 | con: -              | queue=[] | best=12
kết quả: 12


Trace Cách 2 (đi ngược lên có memo) trên cùng ví dụ — in `path` gom được và các giá trị điền vào:

In [16]:
def trace_memo(n, headID, manager, informTime):
    t = [-1] * n
    t[headID] = 0
    for i in range(n):
        path, cur = [], i
        while t[cur] == -1:
            path.append(cur)
            cur = manager[cur]
        for x in reversed(path):
            p = manager[x]
            t[x] = t[p] + informTime[p]
        print(f"i={i}: path chưa biết = {path}, dừng tại nút đã biết {cur} | t = {t}")
    return max(t)

print("kết quả:", trace_memo(n4, 0, mgr4, it4))

i=0: path chưa biết = [], dừng tại nút đã biết 0 | t = [0, -1, -1, -1, -1, -1]
i=1: path chưa biết = [1], dừng tại nút đã biết 0 | t = [0, 4, -1, -1, -1, -1]
i=2: path chưa biết = [2], dừng tại nút đã biết 0 | t = [0, 4, 4, -1, -1, -1]
i=3: path chưa biết = [3], dừng tại nút đã biết 1 | t = [0, 4, 4, 6, -1, -1]
i=4: path chưa biết = [4], dừng tại nút đã biết 2 | t = [0, 4, 4, 6, 5, -1]
i=5: path chưa biết = [5], dừng tại nút đã biết 4 | t = [0, 4, 4, 6, 5, 12]
kết quả: 12


Quan sát: khi `i = 3`, `path = [3]` dừng ngay ở `1` (đã biết từ trước) — không đi lại tới gốc. Đây là chỗ memo tiết kiệm công.

## 9. Code nộp LeetCode

In [17]:
from collections import deque

class Solution:
    def numOfMinutes(self, n: int, headID: int, manager: list[int], informTime: list[int]) -> int:
        children = [[] for _ in range(n)]
        for i, p in enumerate(manager):
            if p != -1:
                children[p].append(i)
        t = [0] * n                    # t[i]: thời điểm i nhận tin
        q = deque([headID])
        best = 0
        while q:
            u = q.popleft()
            best = max(best, t[u])
            for v in children[u]:
                t[v] = t[u] + informTime[u]
                q.append(v)
        return best

for n, head, mgr, it, want in EXAMPLES:
    got = Solution().numOfMinutes(n, head, mgr, it)
    print(got, "OK" if got == want else "SAI")

0 OK
1 OK
21 OK
3 OK


## 10. Vì sao thuật toán đúng

**Bổ đề:** `t[i]` (thời điểm `i` nhận tin) = tổng `informTime` của các nút trên đường từ `headID` tới **cha** của `i`.

*Quy nạp theo độ sâu:* `t[headID] = 0` đúng. Nếu `p = manager[i]` nhận tin lúc `t[p]`, theo đề, sau đúng `informTime[p]` phút mọi cấp dưới trực tiếp (trong đó có `i`) nhận tin ⇒ `t[i] = t[p] + informTime[p]`. Vì cấu trúc là cây, `i` chỉ có **một** cha nên không có cách nào nhận tin sớm hơn.

Mọi người biết tin vào lúc `max_i t[i]`. Cách 2, 3 (top-down), 4 đều tính đúng công thức này theo thứ tự cha trước con (Cách 2 nhờ điền ngược `path`). Cách 3 bản đệ quy dùng dạng tương đương `f(u) = informTime[u] + max f(con)`, và `f(headID) = max t[i]` (quy nạp từ lá lên: `f(u)` = thời gian, tính từ lúc `u` biết tin, để mọi người trong cây con của `u` biết tin).

## 11. Phân tích Time / Space

Gọi `n` = số nhân viên (≤ `10^5`), `H` = chiều cao cây.

| Cách | Time | Space |
|---|---|---|
| 1. Đi lên không nhớ | `O(n·H)`; worst (dây) `O(n²)` | `O(1)` |
| 2. Đi lên có memo | `O(n)` | `O(n)` (`t` và `path`) |
| 3. DFS từ head | `O(n)` | `O(n)` (`children` + stack) |
| 4. BFS từ head | `O(n)` | `O(n)` |

- Cây "bụi" (gốc có `n−1` con, như Ví dụ 2): `H = 1`, brute force cũng `O(n)`.
- Cây "dây": chỉ Cách 2–4 chạy kịp; bản đệ quy cần tăng giới hạn đệ quy.

## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [18]:
METHODS = {
    "brute": minutes_brute,
    "memo": minutes_memo,
    "dfs": minutes_dfs,
    "bfs": minutes_bfs,
    "submit": lambda n, h, m, it: Solution().numOfMinutes(n, h, m, it),
}
edge = [
    (6, 0, [-1, 0, 0, 1, 2, 4], [4, 2, 1, 0, 7, 0], 12),     # ví dụ mục 4
    (2, 1, [1, -1], [0, 1000], 1000),                         # head không phải 0
    (3, 0, [-1, 0, 0], [0, 0, 0], 0),                         # quản lý có informTime = 0 vẫn hợp lệ
    (4, 2, [2, 2, -1, 0], [5, 0, 3, 0], 8),                   # 2 -> 0 -> 3
    (5, 0, [-1, 0, 1, 0, 3], [1, 10, 0, 2, 0], 11),           # nhánh dài hơn chưa chắc lâu hơn
]
for name, fn in METHODS.items():
    for n, h, m, it, want in EXAMPLES + edge:
        got = fn(n, h, m, it)
        assert got == want, (name, n, h, m, it, got, want)
print("Tất cả", len(METHODS), "cách đúng trên", len(EXAMPLES) + len(edge), "test")

Tất cả 5 cách đúng trên 9 test


### 12.2 Đối chiếu ngẫu nhiên (seed cố định, sinh đúng ràng buộc)

Sinh cây ngẫu nhiên: xáo trộn nhãn, mỗi nút (trừ gốc) chọn cha trong các nút đứng trước. `informTime = 0` cho lá, ngẫu nhiên `1..1000` cho nút có con — đúng như ràng buộc.

In [19]:
def random_company(rng, n):
    labels = list(range(n)); rng.shuffle(labels)
    manager = [-1] * n
    for k in range(1, n):
        if rng.random() < 0.3:
            parent = labels[k - 1]                     # tạo các đoạn dây dài
        else:
            parent = labels[rng.randrange(k)]
        manager[labels[k]] = parent
    has_child = [False] * n
    for i, p in enumerate(manager):
        if p != -1: has_child[p] = True
    informTime = [rng.randint(1, 1000) if has_child[i] else 0 for i in range(n)]
    return n, labels[0], manager, informTime

rng = random.Random(1376)
for _ in range(500):
    n, h, m, it = random_company(rng, rng.randint(1, 60))
    res = {name: fn(n, h, m, it) for name, fn in METHODS.items()}
    assert len(set(res.values())) == 1, res
print("500 công ty ngẫu nhiên: mọi cách khớp nhau")

500 công ty ngẫu nhiên: mọi cách khớp nhau


### 12.3 Test lớn theo ràng buộc (`n = 10^5`): dây dài và cây bụi

In [20]:
N = 100_000
chain_mgr = [i + 1 for i in range(N - 1)] + [-1]
chain_it = [0] + [1000] * (N - 1)
star_mgr = [-1] + [0] * (N - 1)
star_it = [1000] + [0] * (N - 1)
rnd = random_company(random.Random(9), N)
# Bản DFS đệ quy (minutes_dfs) bị loại khỏi test dây 10^5: mỗi tầng tốn 2 khung (hàm + generator)
# nên vượt giới hạn đệ quy — xem Lỗi 4 ở mục 13. Bản nộp Cách 3 dùng stack nên vẫn có mặt (submit).
for name in ["memo", "bfs", "submit"]:
    t0 = time.perf_counter()
    a = METHODS[name](N, N - 1, chain_mgr, chain_it)
    b = METHODS[name](N, 0, star_mgr, star_it)
    c = METHODS[name](*rnd)
    print(f"{name:7s} -> dây {a}, bụi {b}, ngẫu nhiên {c} | {time.perf_counter() - t0:.3f}s")

memo    -> dây 99999000, bụi 1000, ngẫu nhiên 20355 | 0.094s


bfs     -> dây 99999000, bụi 1000, ngẫu nhiên 20355 | 0.206s


submit  -> dây 99999000, bụi 1000, ngẫu nhiên 20355 | 0.203s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Cộng tất cả `informTime` thay vì lấy đường dài nhất

In [21]:
def wrong_sum_all(n, headID, manager, informTime):
    return sum(informTime)                # SAI: các nhánh lan truyền SONG SONG

n, h, m, it, want = (15, 0, [-1, 0, 0, 1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 6], [1] * 7 + [0] * 8, 3)
print("sai:", wrong_sum_all(n, h, m, it), "| đúng:", minutes_bfs(n, h, m, it))

sai: 7 | đúng: 3


**Vì sao sai:** các nhánh khác nhau báo tin **đồng thời**, không nối tiếp. Cây nhị phân 15 nút: tổng 7 nhưng thực tế chỉ 3 phút (mỗi tầng 1 phút).

**Code đúng:** lấy **max** của tổng trên từng đường gốc → lá.

### Lỗi 2 — Cộng `informTime` của chính mình thay vì của quản lý

In [22]:
def wrong_own_time(n, headID, manager, informTime):
    children = build_children(n, manager)
    t = [0] * n
    q, best = deque([headID]), 0
    while q:
        u = q.popleft()
        best = max(best, t[u])
        for v in children[u]:
            t[v] = t[u] + informTime[v]   # SAI: phải là informTime[u]
            q.append(v)
    return best

for n, h, m, it, want in EXAMPLES[1:3]:
    print("sai:", wrong_own_time(n, h, m, it), "| đúng:", want)

sai: 0 | đúng: 1
sai: 20 | đúng: 21


**Vì sao sai:** `informTime[v]` là thời gian **`v` cần để báo cấp dưới của `v`**, không phải thời gian để `v` nhận tin. Ví dụ 2: các nhân viên là lá có `informTime = 0` ⇒ code sai trả 0 thay vì 1 (bỏ mất 1 phút của head).

**Code đúng:** `t[v] = t[u] + informTime[u]`.

### Lỗi 3 — Cộng max `informTime` của từng **tầng**

In [23]:
def wrong_level_max(n, headID, manager, informTime):
    children = build_children(n, manager)
    total, level = 0, [headID]
    while level:
        total += max(informTime[u] for u in level)   # SAI: người chậm nhất tầng có thể không nằm trên đường găng
        level = [v for u in level for v in children[u]]
    return total

n, h, m, it = 6, 0, [-1, 0, 0, 1, 2, 4], [1, 5, 1, 0, 10, 0]
print("sai:", wrong_level_max(n, h, m, it), "| đúng:", minutes_bfs(n, h, m, it))

sai: 16 | đúng: 12


**Vì sao sai:** tầng 1 có người chậm nhất là `1` (5 phút), tầng 2 có `4` (10 phút) — nhưng `1` và `4` **không cùng một đường**. Đường thật sự lâu nhất là `0 → 2 → 4 → 5`: `1 + 1 + 10 = 12`, còn code sai cộng `1 + 5 + 10 = 16`.

**Code đúng:** truyền thời điểm theo **từng đường** (BFS/DFS mang `t`), không gộp theo tầng.

### Lỗi 4 — Đệ quy trên cây dây `10^5` với giới hạn mặc định

Mô phỏng giới hạn mặc định rồi bắt lỗi để in (không để exception văng ra):

In [24]:
old = sys.getrecursionlimit()
sys.setrecursionlimit(1000)
try:
    print(minutes_dfs(5000, 4999, [i + 1 for i in range(4999)] + [-1], [0] + [1] * 4999))
except RecursionError as e:
    print("RecursionError:", str(e)[:45], "...")
finally:
    sys.setrecursionlimit(old)
print("BFS:", minutes_bfs(5000, 4999, [i + 1 for i in range(4999)] + [-1], [0] + [1] * 4999))

RecursionError: maximum recursion depth exceeded ...
BFS: 4999


**Code đúng:** BFS/DFS bằng stack/queue, hoặc Cách 2 (vòng lặp đi lên).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Mảng cha `parent[i]`** + câu hỏi "lan truyền từ gốc" ⇒ dựng `children` rồi BFS/DFS; hoặc dùng memo đi lên nếu chỉ cần giá trị của từng nút.
- "Thời gian để **tất cả** nhận được" trên cây ⇒ **đường có trọng số lớn nhất** từ gốc (không phải tổng).
- Trọng số nằm trên **nút cha** (người báo) chứ không phải nút con — đọc kỹ ai tốn thời gian.
- Cây ⇒ mỗi nút một đường từ gốc ⇒ BFS thường là đủ, **không cần Dijkstra**. Nếu là đồ thị tổng quát có nhiều đường ⇒ Dijkstra (LeetCode 743 Network Delay Time).
- Python + cây `10^5` ⇒ tránh đệ quy hoặc tăng giới hạn.

## 15. Biến thể của bài

1. **LeetCode 743 — Network Delay Time:** đồ thị có hướng có trọng số cạnh, nhiều đường ⇒ Dijkstra, đáp án = max khoảng cách.
2. **Trả về danh sách nhân viên nhận tin cuối cùng** (những `i` có `t[i]` bằng đáp án).
3. **Truy vấn: đến phút `T` có bao nhiêu người biết tin?** ⇒ tính `t`, sắp xếp, `bisect`.
4. **Mỗi quản lý báo lần lượt từng người** (mỗi cuộc gọi mất 1 phút): cần chọn thứ tự báo — sắp con theo thời gian cây con giảm dần (bài "broadcast" kinh điển).
5. **Nhiều người đứng đầu / rừng**: thêm một gốc ảo nối tới các gốc với `informTime = 0`.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Đi lên không nhớ | mỗi nhân viên lần lên gốc, cộng `informTime` quản lý | `O(n·H)` | `O(1)` | kiểm chứng, cây nông |
| 2. Đi lên có memo | điền `t[i]` một lần, dừng khi gặp nút đã biết | `O(n)` | `O(n)` | không muốn dựng `children` |
| 3. DFS từ head | `f(u) = it[u] + max f(con)` hoặc stack top-down | `O(n)` | `O(n)` | quen DFS, cần thông tin cây con |
| 4. BFS từ head | lan truyền `t` theo hàng đợi | `O(n)` | `O(n)` | **khuyên dùng**, mô phỏng tự nhiên |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `n = 4, headID = 0, manager = [-1,0,1,2], informTime = [3,2,1,0]`? Và `n = 4, headID = 3, manager = [3,3,3,-1], informTime = [0,0,0,9]`?

**Bài 2 (tính tay):** với `manager = [-1,0,0,1,1,2]`, `informTime = [2,3,4,0,0,0]`, `headID = 0`, tính mảng `t` (thời điểm nhận tin của từng người).

**Bài 3 (điền TODO):** trả về **danh sách đã sắp** các nhân viên nhận tin **muộn nhất**.

In [25]:
def last_informed(n, headID, manager, informTime):
    """Bài 3 — điền TODO. (Bài tập — chưa gọi)"""
    children = build_children(n, manager)
    t = [0] * n
    q = deque([headID])
    while q:
        u = q.popleft()
        # TODO: lan truyền t cho các con của u
        pass
    # TODO: trả về sorted([i for i in range(n) if t[i] == max(t)])
    return []

**Bài 4 (tự viết lại):** viết `f(u) = informTime[u] + max(f(con))` **không đệ quy**: lấy thứ tự BFS rồi duyệt **ngược** thứ tự đó để tính `f` từ lá lên gốc.

**Bài 5 (mini challenge):** cho thêm danh sách truy vấn `queries` (các mốc phút `T`). Với mỗi `T`, trả về số nhân viên **đã biết tin** tại thời điểm `T` (người nhận tin đúng lúc `T` tính là đã biết).

In [26]:
def informed_by(n, headID, manager, informTime, queries):
    """Bài 5 — tự viết. (Bài tập — chưa gọi)"""
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

| Bài | Gợi ý 1 | Gợi ý 2 | Gợi ý 3 |
|---|---|---|---|
| 1 | Cây thứ nhất là dây | cộng `informTime` các quản lý: 3 + 2 + 1 | 6 và 9 |
| 2 | `t[0] = 0` | con của 0 nhận lúc 2; con của 1 lúc 2 + 3 | `[0, 2, 2, 5, 5, 6]` |
| 3 | Giống Cách 4 | `t[v] = t[u] + informTime[u]; q.append(v)` | lọc những `i` có `t[i] == max(t)` |
| 4 | Thứ tự BFS: cha luôn trước con | duyệt ngược ⇒ con trước cha | `f[u] = it[u] + max(f[v])` nếu có con, ngược lại 0 |
| 5 | Tính `t` trước | sắp xếp `t` | `bisect_right(sorted_t, T)` |

## 19. Đáp án tham khảo cho bài tập

In [27]:
# Bài 1
ans1 = [minutes_bfs(4, 0, [-1, 0, 1, 2], [3, 2, 1, 0]), minutes_bfs(4, 3, [3, 3, 3, -1], [0, 0, 0, 9])]
print("Bài 1:", ans1)
assert ans1 == [6, 9]

# Bài 2
def inform_times(n, headID, manager, informTime):
    children = build_children(n, manager)
    t = [0] * n
    q = deque([headID])
    while q:
        u = q.popleft()
        for v in children[u]:
            t[v] = t[u] + informTime[u]
            q.append(v)
    return t

ans2 = inform_times(6, 0, [-1, 0, 0, 1, 1, 2], [2, 3, 4, 0, 0, 0])
print("Bài 2:", ans2)
assert ans2 == [0, 2, 2, 5, 5, 6]

Bài 1: [6, 9]
Bài 2: [0, 2, 2, 5, 5, 6]


In [28]:
# Bài 3
def last_informed_sol(n, headID, manager, informTime):
    t = inform_times(n, headID, manager, informTime)
    m = max(t)
    return sorted(i for i in range(n) if t[i] == m)

assert last_informed_sol(6, 0, [-1, 0, 0, 1, 2, 4], [4, 2, 1, 0, 7, 0]) == [5]
assert last_informed_sol(6, 2, [2, 2, -1, 2, 2, 2], [0, 0, 1, 0, 0, 0]) == [0, 1, 3, 4, 5]
assert last_informed_sol(1, 0, [-1], [0]) == [0]
print("Bài 3: OK")

Bài 3: OK


In [29]:
# Bài 4
def minutes_bottom_up_iter(n, headID, manager, informTime):
    children = build_children(n, manager)
    order, q = [], deque([headID])
    while q:
        u = q.popleft(); order.append(u); q.extend(children[u])
    f = [0] * n
    for u in reversed(order):                     # con trước cha
        if children[u]:
            f[u] = informTime[u] + max(f[v] for v in children[u])
    return f[headID]

rng = random.Random(4)
for _ in range(300):
    args = random_company(rng, rng.randint(1, 50))
    assert minutes_bottom_up_iter(*args) == minutes_brute(*args)
print("Bài 4: OK")

Bài 4: OK


In [30]:
# Bài 5
from bisect import bisect_right

def informed_by_sol(n, headID, manager, informTime, queries):
    t = sorted(inform_times(n, headID, manager, informTime))
    return [bisect_right(t, T) for T in queries]

res5 = informed_by_sol(6, 0, [-1, 0, 0, 1, 2, 4], [4, 2, 1, 0, 7, 0], [0, 3, 4, 5, 6, 11, 12, 100])
print("Bài 5:", res5)
assert res5 == [1, 1, 3, 4, 5, 5, 6, 6]
rng = random.Random(5)
for _ in range(200):
    args = random_company(rng, rng.randint(1, 30))
    tt = inform_times(*args)
    qs = [rng.randint(0, 3000) for _ in range(5)]
    assert informed_by_sol(*args, qs) == [sum(x <= T for x in tt) for T in qs]
print("Bài 5: OK")

Bài 5: [1, 1, 3, 4, 5, 5, 6, 6]
Bài 5: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template lan truyền từ gốc trên cây dạng mảng cha:**

```python
children = [[] for _ in range(n)]
for i, p in enumerate(parent):
    if p != -1: children[p].append(i)
t = [0] * n
q = deque([root]); best = 0
while q:
    u = q.popleft()
    best = max(best, t[u])
    for v in children[u]:
        t[v] = t[u] + w[u]          # trọng số của NGƯỜI BÁO (cha)
        q.append(v)
return best
```

**Checklist trước khi submit:**
- [ ] Lấy **max** theo đường, không cộng tất cả.
- [ ] Cộng `informTime` của **quản lý** (`u`), không phải của cấp dưới (`v`).
- [ ] Bắt đầu từ `headID` (có thể khác 0).
- [ ] `children = [[] for _ in range(n)]`, không dùng `[[]] * n`.
- [ ] Cây dây `10^5`: không đệ quy (hoặc tăng giới hạn).
- [ ] Thử `n = 1` → 0; cây bụi → `informTime[head]`.
- [ ] Đúng chữ ký: `numOfMinutes(self, n, headID, manager, informTime) -> int`.